# Visualização de resultados — bruno_ssl_noise

Notebook **apenas para exploração/visualização** pós-treino. Todo o treino e a
detecção de ruído são feitos pelos scripts (`ssl_pretrain.py`, `train.py`,
`evaluate.py`, `detect_noise.py`). Aqui só carregamos os artefatos gerados e
plotamos.

Ajuste os caminhos abaixo para a pasta de resultados do seu experimento.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUNS = '../runs'          # pasta raiz dos resultados
EVAL_DIR = f'{RUNS}/eval'
NOISE_DIR = f'{RUNS}/noise'
DATA_DIR = '/data/cassava'   # onde estão train_images/ (para mostrar exemplos)
IMG_DIR = f'{DATA_DIR}/train_images'

## 1. Resumo das métricas (individual vs ensemble)

In [ ]:
summary = pd.read_csv(f'{EVAL_DIR}/summary.csv')
display(summary)

ax = summary.plot(x='model', y=['accuracy', 'macro_f1'], kind='bar', figsize=(7,4))
ax.set_title('Acurácia e macro-F1 por backbone e ensemble')
ax.set_ylim(0, 1); plt.tight_layout(); plt.show()

## 2. Matriz de confusão do ensemble

In [ ]:
cm = pd.read_csv(f'{EVAL_DIR}/ensemble_confusion_matrix.csv', index_col=0)
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(cm.values, cmap='Blues')
ax.set_xticks(range(len(cm.columns))); ax.set_xticklabels(cm.columns, rotation=45, ha='right')
ax.set_yticks(range(len(cm.index))); ax.set_yticklabels(cm.index)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm.values[i, j], ha='center', va='center')
ax.set_xlabel('Predito'); ax.set_ylabel('Verdadeiro'); ax.set_title('Matriz de confusão — ensemble')
plt.colorbar(im); plt.tight_layout(); plt.show()

## 3. Concordância entre os três detectores de ruído

In [ ]:
with open(f'{NOISE_DIR}/agreement.json') as f:
    pairs = json.load(f)
print('Jaccard par a par:'); [print(f'  {k}: {v:.3f}') for k, v in pairs.items()]

votes = pd.read_csv(f'{NOISE_DIR}/votes.csv')
dist = votes['n_votes'].value_counts().sort_index()
dist.plot(kind='bar'); plt.xlabel('Nº de métodos que apontaram a imagem');
plt.ylabel('Qtd de imagens'); plt.title('Consenso entre detectores'); plt.tight_layout(); plt.show()
print('\nImagens apontadas pelos 3 métodos:')
display(votes[votes['n_votes'] == 3].head(20))

## 4. Exemplos visuais das imagens mais suspeitas

Mostra as top imagens do ranking do Confident Learning (troque o CSV para ver
os outros métodos).

In [ ]:
import cv2

susp = pd.read_csv(f'{NOISE_DIR}/cl_suspects.csv').head(9)
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, (_, row) in zip(axes.ravel(), susp.iterrows()):
    img = cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, row['image_id'])), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.axis('off')
    ax.set_title(f"rótulo={row['y_true']} | score={row.get('score_cl', float('nan')):.2f}", fontsize=9)
plt.suptitle('Top-9 rótulos suspeitos (Confident Learning)'); plt.tight_layout(); plt.show()